# Pitch arsenal optimization

Hiring-style walkthrough: train a pitch-outcome model, then recommend usage changes with expected xwOBA impact.

**Demo window:** MLB 2026-04-01 → 2026-05-15 (~176k pitches).

In [ ]:
from pathlib import Path

from pitch_dataset.arsenal import (
    format_recommendation_report,
    load_outcome_model,
    optimize_pitcher,
    train_outcome_model,
)
from pitch_dataset.storage import read_pitches

data_path = Path("../data/pitches_mlb_2026.parquet")
model_path = Path("../models/outcome_model.joblib")
pitches = read_pitches(data_path)
print(len(pitches), pitches["game_date"].min(), pitches["game_date"].max())

In [ ]:
if model_path.exists():
    model = load_outcome_model(model_path)
    print("loaded", model_path, model.meta)
else:
    model, metrics = train_outcome_model(pitches, model_path=model_path)
    print(metrics)

In [ ]:
rec = optimize_pitcher(pitches, model, pitcher="Cease")
print(format_recommendation_report(rec))

## Method (short)

1. **Outcome model:** `HistGradientBoostingRegressor` predicts pitch-level run value (`delta_run_exp`) and an approximate pitch xwOBA from context + pitch type.
2. **Context features:** platoon, count, zone/location, TTO, baserunners, batter prior xwOBA, previous pitch, and arsenal pairing (velo/movement separation, release similarity).
3. **Counterfactuals:** for each pitcher × platoon (and count) segment, solve a constrained mix that minimizes expected xwOBA subject to min/max % and max shift from current usage.
4. **Limitations:** early-season sample; holds location fixed; ignores catcher/game-planning constraints; pitch xwOBA for non-BIP is a heuristic.